<a href="https://colab.research.google.com/github/crezny/DATASCI266_final_project/blob/main/KG Construction/1-Policies to Neo4j.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

This file loads data from data\datasource.csv into the Neo4j database.  this will be used to map ontologies to specific policy sections

In [14]:
%pip install neo4j

Note: you may need to restart the kernel to use updated packages.


In [15]:
os.getenv("NEO4J_URI")

'neo4j://<REDACTED>:7687'

In [16]:
import os
from pathlib import Path
import pandas as pd, hashlib, re, math


from neo4j import GraphDatabase
from datetime import datetime
from typing import Iterable, Dict, List
from sqlalchemy import create_engine, text
from sqlalchemy.exc import SQLAlchemyError
import os
try:
    from google.colab import userdata
    get_secret = userdata.get
except ImportError:
    # Fallback: define a dummy get_secret for local use
    def get_secret(key):
        return None

from dotenv import load_dotenv
load_dotenv()


from sqlalchemy import create_engine, text
from sqlalchemy.exc import SQLAlchemyError

def get_env(key: str, default: str = None):
    """Tries os.getenv first, then Colab userdata if available."""
    return os.getenv(key) or get_secret(key) or default

NEO4J_URI  = get_env("NEO4J_URI")   # bolt:// or neo4j://
NEO4J_USER = get_env("NEO4J_USER")
NEO4J_PASS = get_env("NEO4J_PASS")
NEO4J_DATABASE = get_env("NEO4J_DATABASE")

DB_HOST = get_env("POSTGRES_DB_HOST")
DB_PORT = int(get_env("POSTGRES_DB_PORT"))
DB_NAME = get_env("POSTGRES_DB_NAME")
DB_USER = get_env("POSTGRES_DB_USER")
DB_PASS = get_env("POSTGRES_DB_PASS")

connection_url = f"postgresql+psycopg2://{DB_USER}:{DB_PASS}@{DB_HOST}:{DB_PORT}/{DB_NAME}"

In [17]:
def norm_num(s):
    if pd.isna(s): return None
    s = str(s).strip()
    s = s[:-1] if s.endswith(".") else s
    bullet_rx = re.compile(r"^\s*(?:[-*•]|(?:\([a-z]\)))\s+", re.I|re.M)
# split by bullets, enumerate, create child units with number f"{num}.{chr(96+i)}"

    return s

def clean(t):
    t = re.sub(r"\r\n?", "\n", str(t or ""))
    t = re.sub(r"[ \t]+", " ", t)
    t = re.sub(r"\n{3,}", "\n\n", t)
    return t.strip()


#### Go from policies, Sections/Sub Sections and Lines to nodes/edges to be inserted into neo4J

In [18]:
def test_connection():
    try:
        engine = create_engine(connection_url, echo=False)
        with engine.connect() as conn:
            result = conn.execute(text("SELECT version();"))
            version = result.scalar()
            print("✅ Connection successful!")
            print("PostgreSQL version:", version)
    except SQLAlchemyError as e:
        print("❌ Connection failed:")
        print(e)

test_connection()

✅ Connection successful!
PostgreSQL version: PostgreSQL 16.10 (Ubuntu 16.10-0ubuntu0.24.04.1) on x86_64-pc-linux-gnu, compiled by gcc (Ubuntu 13.3.0-6ubuntu2~24.04) 13.3.0, 64-bit


In [19]:
engine = create_engine(connection_url, echo=False)
df = pd.read_sql('SELECT * FROM policy_lines;', engine)

In [20]:
df["row_no"] = df['row_number']

# 0) Order rows deterministically
df = df.sort_values(["policy_id", "row_no"]).reset_index(drop=True)

# 1) Normalize numbers like "1.", "1)" -> "1"; drop junk like "context"
num_rx = re.compile(r"\d+(?:\.\d+)*$")
def normalize_number(raw):
    if pd.isna(raw): return None
    s = str(raw).strip()
    s = re.sub(r"[.)]$", "", s)  # "1." or "1)" -> "1"
    return s if num_rx.fullmatch(s) else None

df["num_raw"] = df["clause_section_number"].map(normalize_number)

# 2) Attach unnumbered lines to the previous number; Prelude "0" for top matter
df["num"] = df.groupby("policy_id", group_keys=False)["num_raw"].ffill()
df["num"] = df["num"].fillna("0")  # preface / unnumbered

# 3) Clean text
def clean(t):
    t = str(t or "")
    t = re.sub(r"\r\n?", "\n", t)
    t = re.sub(r"[ \t]+", " ", t)
    t = re.sub(r"\n{3,}", "\n\n", t)
    return t.strip()

df["clause_full_text"] = df["clause_full_text"].map(clean)
df["clause_text"]      = df["clause_text"].map(clean)


policies = (
    df.sort_values(["policy_id", "row_no"])
      .groupby("policy_id", as_index=False)
      .agg({
          "policy_title": "first",
          "source_framework": "first",
          "policy_origin": "first",
          "company_id": "first",
          "company_type": "first"
      })
      .rename(columns={
          "policy_title": "title",
          "source_framework": "source_framework",
          "policy_origin": "origin"
      })
      .to_dict("records")
)

# Optional enrichments:
import re
for p in policies:
    pid = p["policy_id"]
    # infer type from title or pid prefix
    # e.g., "Acceptable Use Policy" -> "AUP"
    t = p.get("title","").lower()
    p["type"] = "AUP" if "acceptable use" in t else None

    # infer version from id like CL_0001-AUP-v1.3
    m = re.search(r"-v(\d+(?:\.\d+)*)$", pid)
    p["version"] = m.group(1) if m else None


# 4) Aggregate per (policy_id, num) *preserving row order*
agg_rows = (
    df.groupby(["policy_id","num"], sort=False, dropna=False)
      .agg({
          "policy_title":"first",
          "policy_origin":"first",
          "source_framework":"first",
          "company_id":"first",
          "company_type":"first",
          "section_title":"first",
          "is_heading":"max",
          "row_no": ["min","max", list],
          "clause_full_text": lambda s: "\n".join([x for x in s if x]),
          "clause_text":      lambda s: "\n".join([x for x in s if x]),
      })
      .reset_index()
)

# Flatten multiindex columns
agg_rows.columns = [
    "policy_id","number","policy_title","policy_origin","source_framework",
    "company_id","company_type","section_title","is_heading",
    "first_row","last_row","row_list","full_text","short_text"
]

# 5) Sort key (lexicographic-safe) for number AND a row-based tiebreaker
ALPHA = {chr(97+i): i+1 for i in range(26)}  # a->1
def encode_segment(seg):
    if re.fullmatch(r"\d+", seg):         # numeric
        return f"0{int(seg):06d}"
    if re.fullmatch(r"[a-z]", seg):       # synthetic bullet
        return f"1{ALPHA[seg]:06d}"
    # fallback for odd tokens, but you should have only digits/alpha here
    h = int(hashlib.sha1(seg.encode()).hexdigest()[:6], 16) % 999999
    return f"2{h:06d}"

def sort_key_str(number):
    return ":".join(encode_segment(p) for p in number.split(".")) if number else "0"

# 6) Optional: split bullets into .a .b .c in encounter order using row_no
bullet_line = re.compile(r"^\s*(?:[-*•]|(?:\([a-z]\)))\s+(.*\S)\s*$", re.I)

units = []
lines = []

for _, r in agg_rows.iterrows():
    pid   = r["policy_id"]
    num   = r["number"]
    text  = r["full_text"] or r["short_text"] or ""
    rows  = r["row_list"]

    # build $lines provenance from original df rows
    for rn in rows:
        row = df.loc[df["row_no"] == rn].loc[df["policy_id"] == pid].iloc[0]
        lines.append({
            "unit_hint_number": num,
            "row_no": int(rn),
            "text": row.get("line_text") or row.get("clause_text") or "",
            "policy_id": pid
        })

    matches = [m.group(1) for m in bullet_line.finditer(text)]
    if matches:
        # create child bullets in seen order: a, b, c...
        for i, btxt in enumerate(matches, start=1):
            suffix = chr(96+i)  # 1->a
            child_num = f"{num}.{suffix}"
            units.append({
                "unit_id": hashlib.sha1(f"{pid}::{child_num}::{btxt}".encode()).hexdigest(),
                "policy_id": pid,
                "number": child_num,
                "sort_key_str": sort_key_str(child_num),
                "level": ("CLAUSE" if num.count(".")>=1 else "SUBSECTION"),
                "heading": None,
                "text": btxt,
                "is_heading": False,
                "parent_number": num,
                "first_row": r["first_row"],  # or compute row span per bullet if you want
                "last_row":  r["last_row"],
            })
    else:
        lvl = ("SECTION" if num=="0" or num.count(".")==0
               else "SUBSECTION" if num.count(".")==1
               else "CLAUSE")
        thash = hashlib.sha1(text.encode()).hexdigest()[:8]
        units.append({
            "unit_id": hashlib.sha1(f"{pid}::{num}::{thash}".encode()).hexdigest(),
            "policy_id": pid,
            "number": num,
            "sort_key_str": sort_key_str(num),
            "level": lvl,
            "heading": r["section_title"] if r["is_heading"] else None,
            "text": text,
            "is_heading": bool(r["is_heading"]),
            "parent_number": ".".join(num.split(".")[:-1]) or None,
            "first_row": r["first_row"],
            "last_row":  r["last_row"],
        })

# --- Build a (policy_id, number) -> unit_id lookup from finalized units ---
unit_lookup = {(u["policy_id"], u["number"]): u["unit_id"] for u in units}

# Optional: quick bullet detector
bullet_rx = re.compile(r"^\s*(?:[-*•]|(?:\([a-z]\)))\s+", re.I)

# Map each line to its unit_id and add useful fields
final_lines = []
# We'll also track per-unit line index as we go
line_counters = {}

for ln in lines:
    pid   = ln["policy_id"]
    num   = ln["unit_hint_number"]
    uid   = unit_lookup.get((pid, num))  # may be None if the unit became bullets like 1.2.a, 1.2.b

    # If the parent unit split into bullet children (e.g., 1.2 -> 1.2.a/.b/.c),
    # you can either (a) attach lines to the parent (keep uid=None if you chose not to keep parent),
    # or (b) choose a policy: e.g., attach to the parent; bullets have their own new text
    # Here, we'll attach to the parent if available, otherwise leave None
    # (You can also enhance by computing row spans per bullet to attach lines precisely.)

    key = uid or (pid, num)  # fallback bucket
    line_counters[key] = line_counters.get(key, 0) + 1

    text = ln["text"] or ""
    line_hash = hashlib.sha1(text.encode()).hexdigest()[:8]

    final_lines.append({
        "line_id": hashlib.sha1(f"{pid}::{ln['row_no']}::{line_hash}".encode()).hexdigest(),
        "policy_id": pid,
        "unit_number": num,
        "unit_id": uid,                       # <-- attach to final unit when possible
        "row_no": int(ln["row_no"]),
        "line_index_in_unit": line_counters[key],
        "text": text,
        "line_hash": line_hash,
        "is_bullet": bool(bullet_rx.match(text)),
        "source_number_raw": df.loc[(df["policy_id"]==pid) & (df["row_no"]==ln["row_no"]), "clause_section_number"].iloc[0] if not df.empty else None,
    })

# Replace or keep both versions
lines = final_lines


In [21]:
# ---------- Cypher ----------
# ---------- single-statement queries ----------
CYPHER_CONSTRAINTS = [
    "CREATE CONSTRAINT policy_id IF NOT EXISTS FOR (p:Policy) REQUIRE p.policy_id IS UNIQUE",
    "CREATE CONSTRAINT unit_id   IF NOT EXISTS FOR (u:PolicyUnit) REQUIRE u.unit_id IS UNIQUE",
    "CREATE CONSTRAINT line_id   IF NOT EXISTS FOR (l:Line) REQUIRE l.line_id IS UNIQUE",
    "CREATE INDEX unit_number_performance IF NOT EXISTS FOR (u:PolicyUnit) ON (u.number)",
    "CREATE INDEX unit_sort_key          IF NOT EXISTS FOR (u:PolicyUnit) ON (u.sort_key_str)",
    "CREATE FULLTEXT INDEX unit_text_full IF NOT EXISTS FOR (u:PolicyUnit) ON EACH [u.text, u.heading]",
    "CREATE FULLTEXT INDEX line_text_full IF NOT EXISTS FOR (l:Line) ON EACH [l.text]",
]



# 2) Units + attach to Policy (same single statement)
CYPHER_UPSERT_UNITS = """
WITH $batch AS batch, $ts AS ts
UNWIND batch AS row
MATCH (p:Policy {policy_id: row.policy_id})
MERGE (u:PolicyUnit {unit_id: row.unit_id})
  ON CREATE SET
    u.number        = row.number,
    u.sort_key_str  = row.sort_key_str,
    u.level         = row.level,
    u.heading       = row.heading,
    u.text          = row.text,
    u.is_heading    = coalesce(row.is_heading, false),
    u.first_row     = toInteger(coalesce(row.first_row, 0)),
    u.last_row      = toInteger(coalesce(row.last_row, 0)),
    u.createdAt     = ts,
    u.label         = coalesce(row.label, row.heading, left(row.text,80))
  ON MATCH SET
    u.sort_key_str  = coalesce(row.sort_key_str, u.sort_key_str),
    u.level         = coalesce(row.level, u.level),
    u.heading       = coalesce(row.heading, u.heading),
    u.text          = coalesce(row.text, u.text),
    u.is_heading    = coalesce(row.is_heading, u.is_heading),
    u.first_row     = coalesce(toInteger(row.first_row), u.first_row),
    u.last_row      = coalesce(toInteger(row.last_row),  u.last_row),
    u.label         = coalesce(row.label, u.label)
MERGE (p)-[:HAS_UNIT]->(u)
"""

# 3) Wire hierarchy by parent_number (still one statement)
CYPHER_WIRE_HIERARCHY = """
WITH $batch AS batch
UNWIND batch AS row
WITH row
WHERE row.parent_number IS NOT NULL AND row.parent_number <> ""
MATCH (p:Policy {policy_id: row.policy_id})
MATCH (child:PolicyUnit {unit_id: row.unit_id})
MATCH (p)-[:HAS_UNIT]->(parent:PolicyUnit {number: row.parent_number})
MERGE (parent)-[:HAS_CHILD]->(child)
"""

# 4) Lines + attach to units (prefer unit_id; fallback policy+unit_number) — one statement
CYPHER_UPSERT_LINES = """
WITH $batch AS batch, $ts AS ts
UNWIND batch AS l
MERGE (line:Line {line_id: l.line_id})
  ON CREATE SET
    line.row_no             = toInteger(coalesce(l.row_no, 0)),
    line.line_index_in_unit = toInteger(coalesce(l.line_index_in_unit, 0)),
    line.text               = coalesce(l.text, ""),
    line.line_hash          = coalesce(l.line_hash, ""),
    line.is_bullet          = coalesce(l.is_bullet, false),
    line.source_number_raw  = l.source_number_raw,
    line.createdAt          = ts,
    line.label              = coalesce(l.label, left(l.text,80)),
    line.sort_order         = toInteger(coalesce(l.row_no, 0))  // optional for ordering
  ON MATCH SET
    line.text               = coalesce(l.text, line.text)

WITH l, line
OPTIONAL MATCH (u1:PolicyUnit {unit_id: l.unit_id})
OPTIONAL MATCH (p:Policy {policy_id: l.policy_id})-[:HAS_UNIT]->(u2:PolicyUnit {number: l.unit_number})
WITH line, coalesce(u1, u2) AS u
FOREACH (_ IN CASE WHEN u IS NOT NULL THEN [1] ELSE [] END |
  MERGE (u)-[:HAS_LINE]->(line)
)
"""

CYPHER_UPSERT_POLICIES_FROM_LIST = """
WITH $batch AS batch, $ts AS ts
UNWIND batch AS row
MERGE (p:Policy {policy_id: row.policy_id})
  ON CREATE SET
    p.title           = row.title,
    p.type            = row.type,
    p.version         = row.version,
    p.source_framework= row.source_framework,
    p.origin          = row.origin,
    p.company_id      = row.company_id,
    p.company_type    = row.company_type,
    p.createdAt       = ts,
    p.label           = coalesce(row.label, row.title, row.policy_id)
  ON MATCH SET
    p.title            = coalesce(row.title, p.title),
    p.type             = coalesce(row.type, p.type),
    p.version          = coalesce(row.version, p.version),
    p.source_framework = coalesce(row.source_framework, p.source_framework),
    p.origin           = coalesce(row.origin, p.origin),
    p.company_id       = coalesce(row.company_id, p.company_id),
    p.company_type     = coalesce(row.company_type, p.company_type),
    p.label            = coalesce(row.label, p.label)

WITH p, row
WHERE row.source_framework IS NOT NULL

// Only link if a matching Framework node already exists
OPTIONAL MATCH (f:Framework {name: row.source_framework})

FOREACH (_ IN CASE WHEN f IS NOT NULL THEN [1] ELSE [] END |
  MERGE (p)-[:USES_FRAMEWORK]->(f)
)

// normalized title used for matching EvidenceRequirements
WITH p
SET p.norm_title = toLower(
  apoc.text.replace(p.title, '[^a-zA-Z0-9]+', '_')
);

"""

In [22]:
# ---------- Helpers ----------

def _chunks(seq: List[dict], size: int = 1000) -> Iterable[List[dict]]:
    for i in range(0, len(seq), size):
        yield seq[i:i+size]

def smoke_test(session):
    # Must succeed; if THIS throws the same error, your call site is concatenating strings.
    session.run("UNWIND range(1,3) AS x RETURN sum(x) AS s").single()

def enrich_labels(policies, units, lines):
    """
    Adds a human-readable 'label' property for visualization/search
    to every node type before loading to Neo4j.
    """

    # --- Policies ---
    for p in policies:
        title = (p.get("title") or p.get("policy_title") or "").strip()
        pid = p.get("policy_id", "")
        version = p.get("version", "")
        p["label"] = f"{title} {('v' + version) if version else ''}".strip() or pid

    # --- PolicyUnits ---
    for u in units:
        heading = (u.get("heading") or "").strip()
        number = (u.get("number") or "").strip()
        text = (u.get("text") or "").strip()
        short_text = text.split("\n", 1)[0][:60]
        label = heading or f"{number} {short_text}"
        u["label"] = label.strip()

    # --- Lines ---
    for l in lines:
        text = (l.get("text") or "").strip().split("\n", 1)[0]
        l["label"] = text[:80]



In [23]:
def load_policy_graph(uri, user, password, policies, units, lines, database="neo4j", wipe=False, batch_size=1000):
    driver = GraphDatabase.driver(uri, auth=(user, password))
    try:
        with driver.session(database="system") as sys_sess:
            sys_sess.run(f"CREATE DATABASE `{database}` IF NOT EXISTS WAIT")
        with driver.session(database=database) as sess:
            if wipe:
                sess.run("MATCH (n) DETACH DELETE n")

            # constraints (same as before) ...
            for stmt in CYPHER_CONSTRAINTS:
                sess.run(stmt)

            ts = datetime.utcnow().isoformat()

            # --- load policies first ---
            for batch in _chunks(policies, batch_size):
                sess.run(CYPHER_UPSERT_POLICIES_FROM_LIST, batch=batch, ts=ts)

            # --- then units, hierarchy, lines (same queries as you have) ---
            for batch in _chunks(units, batch_size):
                sess.run(CYPHER_UPSERT_UNITS, batch=batch, ts=ts)

            if any(u.get("parent_number") for u in units):
                for batch in _chunks([u for u in units if u.get("parent_number")], batch_size):
                    sess.run(CYPHER_WIRE_HIERARCHY, batch=batch)

            if lines:
                for batch in _chunks(lines, batch_size):
                    sess.run(CYPHER_UPSERT_LINES, batch=batch, ts=ts)
    finally:
        driver.close()



#### Load into database

In [24]:
enrich_labels(policies, units, lines)
load_policy_graph(
    NEO4J_URI
    , NEO4J_USER
    , NEO4J_PASS
    , units=units, lines=lines, policies=policies
    , database=NEO4J_DATABASE
    , wipe=True
)



In [25]:
policies

[{'policy_id': 'CL_0001-ASMP-v1.0',
  'title': 'Asset Management Policy',
  'source_framework': 'NIST CSF',
  'origin': 'client',
  'company_id': 'CL_0001',
  'company_type': 'Client',
  'type': None,
  'version': '1.0',
  'label': 'Asset Management Policy v1.0'},
 {'policy_id': 'CL_0001-AUP-v2',
  'title': 'Acceptable Use Policy',
  'source_framework': 'NIST CSF',
  'origin': 'client',
  'company_id': 'CL_0001',
  'company_type': 'Client',
  'type': 'AUP',
  'version': '2',
  'label': 'Acceptable Use Policy v2'},
 {'policy_id': 'CL_0001-FWID-v1.0',
  'title': 'Firewall and Intrusion Detection Policy',
  'source_framework': 'NIST CSF',
  'origin': 'client',
  'company_id': 'CL_0001',
  'company_type': 'Client',
  'type': None,
  'version': '1.0',
  'label': 'Firewall and Intrusion Detection Policy v1.0'},
 {'policy_id': 'CL_0001-ITP-v3.0',
  'title': 'Information Technology Policy',
  'source_framework': 'NIST CSF',
  'origin': 'client',
  'company_id': 'CL_0001',
  'company_type': 'Cl